# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path # lam viec voi duong dan file va thu muc
from scipy import stats # cung cap cac cong cu thong ke

sns.set_style('whitegrid') # thiet lap style mac dinh cho bieu do, nen trang vaf cac duong luoi

csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()   # xem 5 dong du lieu dau tien

Matplotlib is building the font cache; this may take a moment.


Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [5]:
# TODO
# xem kich thuoc du lieu
print(f"Data size: {df.shape[0]} row, {df.shape[1]} columns")


Data size: 1338 row, 7 columns


In [7]:
print("ten cac cot:") 
", ".join(df.columns)

ten cac cot:


'age, sex, bmi, children, smoker, region, charges'

In [13]:
print("data types:")
df.dtypes

data types:


age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

## A.2. Missing values & Duplicate data

In [20]:
# TODO
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [19]:
print("So gia tri bi trung: ", df.duplicated().sum())

So gia tri bi trung:  1


## A.3. Invalid values

In [ ]:
# TODO
# gia tri khong hop le VD: age khong dc <0
print("age:", (df['age'] <= 0).sum())
print("bmi:", (df['bmi'] <= 0).sum())
print("children:", (df['children'] < 0).sum())  # so con
print("charges:", (df['charges'] <= 0).sum())

#????
print((~df['sex'].isin(['male', 'female'])).sum())
print((~df['smoker'].isin(['yes', 'no'])).sum())

age: 0
bmi: 0
children: 0
charges: 0
0
0


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [26]:
import numpy as np

In [29]:
# TODO
conditions = [
    df['bmi'] < 25,
    (df['bmi'] >= 25) & (df['bmi'] < 30),
    df['bmi'] >= 30
]

choices = ['Normal', 'Overweight', 'Obese']

df['bmi_group'] = np.select(conditions, choices, default='Unknown')

---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [ ]:
# TODO
# gia tri trung binh - mean
df[['age', 'bmi', 'children', 'charges']].mean()

age            39.207025
bmi            30.663397
children        1.094918
charges     13270.422265
dtype: float64

In [31]:
# gia tri trung vi - median
df[['age', 'bmi', 'children', 'charges']].median()

age           39.000
bmi           30.400
children       1.000
charges     9382.033
dtype: float64

In [33]:
# gia tri yeu vi - mode
df[['age', 'bmi', 'children', 'charges']].mode().iloc[0]

age           18.0000
bmi           32.3000
children       0.0000
charges     1639.5631
Name: 0, dtype: float64

## Group 2 — Dispersion (do phan tan)

In [34]:
# TODO
num_cols = ['age', 'bmi', 'children', 'charges']

# range
df[num_cols].max() - df[num_cols].min()

age            46.00000
bmi            37.17000
children        5.00000
charges     62648.55411
dtype: float64

In [35]:
# var
df[num_cols].var()

age         1.974014e+02
bmi         3.718788e+01
children    1.453213e+00
charges     1.466524e+08
dtype: float64

In [36]:
# std
df[num_cols].std()

age            14.049960
bmi             6.098187
children        1.205493
charges     12110.011237
dtype: float64

## Group 3 — Location and Shape

In [37]:
# TODO
# vi tri
df[['age', 'bmi', 'children', 'charges']].quantile([0.25, 0.5, 0.75])

,age,bmi,children,charges
0.25,27.0,26.29625,0.0,4740.287150
0.50,39.0,30.40000,1.0,9382.033000
0.75,51.0,34.69375,2.0,16639.912515


In [38]:
# hinh dang
df[['age', 'bmi', 'children', 'charges']].skew()

age         0.055673
bmi         0.284047
children    0.938380
charges     1.515880
dtype: float64

---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [39]:
# TODO
df.groupby('smoker')['charges'].mean()

df.groupby(['region', 'smoker'])['charges'].mean().unstack()

region_compare = df.groupby(['region', 'smoker'])['charges'].mean().unstack()

region_compare['ratio'] = (
    region_compare['yes'] / region_compare['no']
)

region_compare

smoker,no,yes,ratio
region,,,
northeast,9165.531672,29673.536473,3.237514
northwest,8556.463715,30192.003182,3.528561
southeast,8032.216309,34844.996824,4.338155
southwest,8019.284513,32269.063494,4.023933


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [41]:
# TODO
df.groupby('smoker')[['bmi', 'charges']].corr()
smoker_corr = df.groupby('smoker')[['bmi', 'charges']].corr().iloc[0::2, -1]


print(smoker_corr)

smoker     
no      bmi    0.084037
yes     bmi    0.806481
Name: charges, dtype: float64


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [8]:
# TODO
region_charges = df.groupby('region')['charges'].agg(
    Trung_binh='mean',
    Trung_vi='median',
    So_luong='count'
).sort_values(by='Trung_binh', ascending=False)

print("=== CHI PHÍ BẢO HIỂM THEO VÙNG ===")
print(region_charges)

=== CHI PHÍ BẢO HIỂM THEO VÙNG ===
             Trung_binh      Trung_vi  So_luong
region                                         
southeast  14735.411438   9294.131950       364
northeast  13406.384516  10057.652025       324
northwest  12417.575374   8965.795750       325
southwest  12346.937377   8798.593000       325


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [9]:
# TODO
children_stats = df.groupby('children')['charges'].agg(
    Trung_binh='mean',
    Trung_vi='median',
    So_luong='count'
)

print("=== CHI PHÍ BẢO HIỂM THEO SỐ CON ===")
print(children_stats)

corr = df['children'].corr(df['charges'])
print(f"\nHệ số tương quan (Correlation): {corr:.3f}")

=== CHI PHÍ BẢO HIỂM THEO SỐ CON ===
            Trung_binh     Trung_vi  So_luong
children                                     
0         12365.975602   9856.95190       574
1         12731.171832   8483.87015       324
2         15073.563734   9264.97915       240
3         15355.318367  10600.54830       157
4         13850.656311  11033.66170        25
5          8786.035247   8589.56505        18

Hệ số tương quan (Correlation): 0.068


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [10]:
# TODO
corr = df['age'].corr(df['charges'])
print(f"Hệ số tương quan (Pearson): {corr:.3f}")

df['age_group'] = pd.cut(df['age'], bins=[17, 30, 45, 65], labels=['18-30', '31-45', '46-64'])

print("\n=== CHI PHÍ BẢO HIỂM TRUNG BÌNH THEO NHÓM TUỔI ===")
print(df.groupby('age_group')['charges'].agg(
    Trung_binh='mean',
    Trung_vi='median',
    So_luong='count'
))

Hệ số tương quan (Pearson): 0.299

=== CHI PHÍ BẢO HIỂM TRUNG BÌNH THEO NHÓM TUỔI ===
             Trung_binh      Trung_vi  So_luong
age_group                                      
18-30       9397.552051   3392.671000       444
31-45      12647.455654   7146.167725       394
46-64      17200.428704  12136.096375       500


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*